# 基于 PyTorch 的 ResNet 迁移学习：生活垃圾图片 40 分类

**任务目标：**

对生活垃圾图片进行 40 个细分类别的识别（一级类别 4 类：可回收物 / 厨余垃圾 / 有害垃圾 / 其他垃圾）。

**思路：**
1. 数据集仅约 1.4 万张图片、类别不平衡，从零训练大模型容易过拟合，所以采用 **ImageNet 预训练 ResNet50 + 微调** 的迁移学习方案；
2. 训练集做随机裁剪 / 翻转 / 颜色抖动等数据增强，验证集 / 测试集仅做 Resize + CenterCrop；
3. 训练完成后，按 `testpath.txt` 中的顺序对 400 张测试图像做预测，写入 `result.txt`，格式：`图像名 标签\n`。

In [ ]:
# 导入必要的库
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as Data
import torchvision
from PIL import Image
from torchvision import transforms

# 固定随机种子，保证结果可复现
SEED: int = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 选择训练设备：优先 CUDA -> MPS (Apple Silicon) -> CPU
device: torch.device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("PyTorch 版本:", torch.__version__)
print("torchvision 版本:", torchvision.__version__)
print("使用设备:", device)

## 1. 数据准备

数据目录结构：

```
data/garbage/
  ├── garbage_dict.json      # 标签 -> 类别名（中文）的映射
  ├── testpath.txt           # 测试图片名顺序
  ├── train/                 # 训练集，按类别分子目录 0..39/
  │   ├── 0/img_*.jpg
  │   └── ...
  └── test/                  # 测试集，400 张图片
      └── test*.jpg
```

下面先看一下数据规模与类别分布。

In [ ]:
# 数据根目录（相对 notebook 所在目录）
DATA_ROOT: Path = Path("../data/garbage")
TRAIN_DIR: Path = DATA_ROOT / "train"
TEST_DIR: Path = DATA_ROOT / "test"
TESTPATH_FILE: Path = DATA_ROOT / "testpath.txt"
DICT_FILE: Path = DATA_ROOT / "garbage_dict.json"
RESULT_FILE: Path = Path("../result.txt")  # 最终提交文件放到项目根目录

# 加载类别字典：{"0": "其他垃圾/一次性快餐盒", ...}
with open(DICT_FILE, "r", encoding="utf-8") as f:
    garbage_dict: dict[str, str] = json.load(f)

NUM_CLASSES: int = len(garbage_dict)
print(f"类别数: {NUM_CLASSES}")
print("前 5 个类别:")
for k in list(garbage_dict)[:5]:
    print(f"  {k} -> {garbage_dict[k]}")

# 统计每个类别的训练样本数
class_dirs: list[Path] = sorted(
    [p for p in TRAIN_DIR.iterdir() if p.is_dir()],
    key=lambda p: int(p.name),
)
class_counts: dict[int, int] = {
    int(p.name): len(list(p.glob("*.jpg"))) for p in class_dirs
}
print(f"\n训练集图像总数: {sum(class_counts.values())}")
print(f"最少样本类别: {min(class_counts.items(), key=lambda kv: kv[1])}")
print(f"最多样本类别: {max(class_counts.items(), key=lambda kv: kv[1])}")

## 2. 构建 Dataset 与 DataLoader

- 训练 / 验证集：使用 `torchvision.datasets.ImageFolder` 加载，目录名即标签字符串。
  注意 `ImageFolder` 默认按字典序排序子目录，会把 `"10"` 排在 `"2"` 之前，
  因此 `ImageFolder` 给出的 `class_idx` **不一定等于子目录名对应的整数**。
  我们后面会建立 `imagefolder_idx -> int(label)` 的映射来还原原始标签。
- 测试集：自定义 Dataset，按 `testpath.txt` 中的顺序读取。

In [ ]:
# ImageNet 上预训练 ResNet 期望的输入尺寸与归一化参数
IMG_SIZE: int = 224
IMAGENET_MEAN: tuple[float, float, float] = (0.485, 0.456, 0.406)
IMAGENET_STD: tuple[float, float, float] = (0.229, 0.224, 0.225)

# 训练集做较强的数据增强，缓解过拟合
train_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

# 验证 / 测试集仅做确定性的预处理
eval_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

# 用 ImageFolder 加载全部训练数据（先不加 transform，等 split 之后再分别赋值）
full_dataset = torchvision.datasets.ImageFolder(root=str(TRAIN_DIR))
print(f"ImageFolder 共加载 {len(full_dataset)} 张图片")
print(f"ImageFolder 的 class_to_idx 前 5 项: {list(full_dataset.class_to_idx.items())[:5]}")

# 建立 ImageFolder 内部索引 -> 原始整数标签 的映射
idx_to_label: dict[int, int] = {v: int(k) for k, v in full_dataset.class_to_idx.items()}
print(f"映射示例 (idx -> 原始标签): {dict(list(idx_to_label.items())[:5])}")

In [ ]:
# 包装一层，让同一份样本在 train/val 子集中使用不同的 transform
class TransformSubset(Data.Dataset):
    def __init__(self, subset: Data.Subset, transform: transforms.Compose) -> None:
        self.subset = subset
        self.transform = transform

    def __len__(self) -> int:
        return len(self.subset)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        img, label = self.subset[idx]  # img 仍是 PIL.Image
        return self.transform(img), label


# 按 9:1 切分训练 / 验证集
val_ratio: float = 0.1
n_total: int = len(full_dataset)
n_val: int = int(n_total * val_ratio)
n_train: int = n_total - n_val

generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = Data.random_split(full_dataset, [n_train, n_val], generator=generator)

train_dataset = TransformSubset(train_subset, train_transform)
val_dataset = TransformSubset(val_subset, eval_transform)

print(f"训练样本: {len(train_dataset)}, 验证样本: {len(val_dataset)}")

# DataLoader
# num_workers=0：notebook 中自定义的 Dataset 类无法被 spawn 出来的子进程 pickle，
# 因此使用主进程加载（MPS 上多 worker 也几乎没收益）
BATCH_SIZE: int = 64
NUM_WORKERS: int = 0

train_loader = Data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = Data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

# 看一个 batch 的形状
xb, yb = next(iter(train_loader))
print(f"一个 batch 形状: X={tuple(xb.shape)}, y={tuple(yb.shape)}")

In [ ]:
# 测试集 Dataset：严格按 testpath.txt 中的顺序加载
class TestImageDataset(Data.Dataset):
    def __init__(self, test_dir: Path, name_list: list[str], transform: transforms.Compose) -> None:
        self.test_dir = test_dir
        self.names = name_list
        self.transform = transform

    def __len__(self) -> int:
        return len(self.names)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, str]:
        name = self.names[idx]
        img = Image.open(self.test_dir / name).convert("RGB")
        return self.transform(img), name


# 读取测试图片名列表（保留原顺序）
with open(TESTPATH_FILE, "r", encoding="utf-8") as f:
    test_names: list[str] = [line.strip() for line in f if line.strip()]
print(f"测试集图片数: {len(test_names)} (前 3 个: {test_names[:3]})")

test_dataset = TestImageDataset(TEST_DIR, test_names, eval_transform)
test_loader = Data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights


def build_model(num_classes: int) -> nn.Module:
    weights = ResNet50_Weights.IMAGENET1K_V2  # 较新的训练权重，验证集表现更好
    model = resnet50(weights=weights)
    # 替换最后的全连接层
    in_features: int = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


model: nn.Module = build_model(NUM_CLASSES).to(device)
# 仅打印最后一层确认替换成功
print(model.fc)

In [ ]:
NUM_EPOCHS: int = 8

criterion: nn.CrossEntropyLoss = nn.CrossEntropyLoss()

backbone_params = [p for n, p in model.named_parameters() if not n.startswith("fc.")]
head_params = [p for n, p in model.named_parameters() if n.startswith("fc.")]

optimizer: optim.AdamW = optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-4},
        {"params": head_params, "lr": 1e-3},
    ],
    weight_decay=1e-4,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(optimizer)

In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: Data.DataLoader) -> tuple[float, float]:
    model.eval()
    loss_sum: float = 0.0
    correct: int = 0
    n: int = 0
    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(X)
        loss = criterion(logits, y)
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += y.size(0)
    return loss_sum / n, correct / n


CKPT_PATH: Path = Path("../data/garbage/resnet50_best.pt")
best_val_acc: float = 0.0
history: dict[str, list[float]] = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss_sum: float = 0.0
    train_correct: int = 0
    n_seen: int = 0

    for X, y in train_loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * y.size(0)
        train_correct += (logits.argmax(dim=1) == y).sum().item()
        n_seen += y.size(0)

    scheduler.step()

    train_loss = train_loss_sum / n_seen
    train_acc = train_correct / n_seen
    val_loss, val_acc = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    flag: str = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CKPT_PATH)
        flag = "  <- 保存当前最优"

    print(
        f"epoch {epoch:2d}/{NUM_EPOCHS} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f}{flag}"
    )

print(f"\n最佳验证准确率: {best_val_acc:.4f}")

## 6. 在测试集上推理并生成 `result.txt`

加载验证集上最优的权重，对 400 张测试图片做预测，按 `testpath.txt` 中的顺序写入 `result.txt`。

输出格式（每行）：

```
图像名<空格>标签
test1.jpg 29
```

In [ ]:
# 抽取 8 张测试图片，展示预测结果
sample_idxs = list(range(8))
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, i in zip(axes.flat, sample_idxs):
    name, label = predictions[i]
    img = Image.open(TEST_DIR / name).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{name}\n-> {label}: {garbage_dict[str(label)]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. 可视化测试集预测样例

In [ ]:
# 加载验证集上最佳的权重做推理
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

predictions: list[tuple[str, int]] = []
with torch.no_grad():
    for X, names in test_loader:
        X = X.to(device, non_blocking=True)
        logits = model(X)
        # 把 ImageFolder 的内部索引映射回原始整数标签
        pred_idx = logits.argmax(dim=1).cpu().tolist()
        for name, idx in zip(names, pred_idx):
            predictions.append((name, idx_to_label[idx]))

print(f"完成推理，共 {len(predictions)} 条预测")
print("前 5 条预测:")
for name, label in predictions[:5]:
    print(f"  {name} -> {label} ({garbage_dict[str(label)]})")

# 验证预测顺序与 testpath.txt 一致
assert [n for n, _ in predictions] == test_names, "预测顺序与 testpath.txt 不一致！"

# 写入 result.txt：格式严格按照 sprintln("%s\t%d", name, label)
with open(RESULT_FILE, "w", encoding="utf-8") as f:
    for name, label in predictions:
        f.write(f"{name}\t{label}\n")

# 自检：行数是否为 400 + 抽几行展示
with open(RESULT_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()
print(f"\nresult.txt 路径: {RESULT_FILE.resolve()}")
print(f"总行数: {len(lines)}")
assert len(lines) == 400, "结果文件必须为 400 行！"
print("\n前 10 行示例:")
print("".join(lines[:10]))

In [ ]:
# 训练曲线可视化
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"], "s-", label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train_acc"], "o-", label="train")
axes[1].plot(epochs, history["val_acc"], "s-", label="val")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 训练 & 验证

每个 epoch 结束后在验证集上计算准确率，并保存验证集表现最好的模型权重作为最终用于推理的模型。

## 4. 损失函数与优化器

- 损失：`nn.CrossEntropyLoss`，多分类标准选择；
- 优化器：分组学习率 —— 骨干网络用较小的 `1e-4`，新加的全连接层用 `1e-3`；
- 学习率调度：`CosineAnnealingLR`，平滑衰减到接近 0。

## 3. 模型：ImageNet 预训练 ResNet50

直接换掉最后的全连接层，输出维度改为 40。骨干网络的权重也参与训练（使用较小的学习率），让特征更贴合垃圾图片这一新域。